In [ ]:
!pip install git+https://github.com/huggingface/peft.git -Uqqq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!pip install bitsandbytes einops -Uqqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 8.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from peft import PeftModel
from transformers import GenerationConfig
import torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
from peft import prepare_model_for_kbit_training

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
model_name = "NousResearch/Llama-2-7b-chat-hf"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,            # load model in 4-bit precision
    bnb_4bit_quant_type="nf4",    # pre-trained model should be quantized in 4-bit NF format
    bnb_4bit_use_double_quant=True, # Using double quantization as mentioned in QLoRA paper
    bnb_4bit_compute_dtype=torch.bfloat16, # During computation, pre-trained model should be loaded in BF16 format
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config, # Use bitsandbytes config
    device_map="auto",  # Specifying device_map="auto" so that HF Accelerate will determine which GPU to put each layer of the model on
    trust_remote_code=True, # Set trust_remote_code=True to use falcon-7b model with custom code
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Setting pad_token same as eos_token

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [ ]:
def generate_llm_response(query, model, tokenizer, device='cuda', max_new_tokens=256):
    """
    Generates a response from an LLM model given a query.

    Args:
        query (str): The input query to be processed by the model.
        model (PreTrainedModel): The LLM model used for generating responses.
        tokenizer (PreTrainedTokenizer): The tokenizer associated with the model.
        device (str): The device to run the model on ('cuda' or 'cpu').
        max_new_tokens (int): Maximum number of tokens to generate in the response.

    Returns:
        str: The generated response text.
    """

    # Define the system and user prompts
    system_prompt = (
        "Answer the following question truthfully.\n"
        "If you don't know the answer, respond 'Sorry, I don't know the answer to this question.'.\n"
        "If the question is too complex, respond 'Kindly, consult a pharmacist for further queries.'."
    )

    user_prompt = f"<HUMAN>: {query}\n<ASSISTANT>: "
    final_prompt = system_prompt + "\n" + user_prompt

    # Tokenize the input prompt
    inputs = tokenizer(final_prompt, return_tensors="pt").to(device)

    # Configure generation parameters
    generation_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        temperature=0.4,
        top_p=0.6,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=False  # Disable caching due to compatibility with gradient checkpointing
    )

    # Generate the response
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            generation_config=generation_config
        )

    # Decode the generated tokens to get the response text
    response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response_text


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
peft_model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/GUARDRAIL/Model/LlamaModel2").to(device)

In [ ]:
query = input("Enter query: ")

response = generate_llm_response(query, peft_model, tokenizer, device=device)
print("Fine-Tuned Model Response:", response)

Enter query: can you test positive from having the hep b vaccine
Fine-Tuned Model Response: Answer the following question truthfully.
If you don't know the answer, respond 'Sorry, I don't know the answer to this question.'.
If the question is too complex, respond 'Kindly, consult a pharmacist for further queries.'.
<HUMAN>: can you test positive from having the hep b vaccine
<ASSISTANT>:  no it will not cause an allergic reaction and should be fine if your doctor gave one i am assuming that they did since there are many different types of flu shots but again just ask him or her <positive_smiley>
